# Session 2 — Does a keyword pattern measure the mechanism it is named after?

**Learning objectives:** run and read static-analysis output and code (objective 2, reading part), and validate a pattern-based indicator against manual inspection (objective 5). About 100 minutes.

The model in Session 1 used twelve regular-expression indicators such as `has_max_tx` ("maximum-transaction limit") and `has_blacklist`. A name is not evidence. In this session you annotate contracts yourself and then measure how often the patterns agree with a careful human reading.

The codebook used in the paper is in `annotation/regex_annotation_sheet.xlsx` (tabs `codebook`, `rule_decisions`). Today you use four of its items.

In [ ]:
# Setup: works locally (run from the teaching/ folder) and in Google Colab.
import os, sys, subprocess
from pathlib import Path
if 'google.colab' in sys.modules and not Path('rugpull-static-detection').exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/fazaaf/rugpull-static-detection.git'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'xgboost==3.4.1', 'shap==0.54.1', 'scikit-learn==1.6.1', 'openpyxl'], check=True)
ROOT = next(p for p in [Path.cwd(), Path.cwd().parent, Path('rugpull-static-detection')] if (p / 'data' / 'D_single_era_matched.csv').exists())
print('repository root:', ROOT.resolve())

In [ ]:
import re, numpy as np, pandas as pd
CON = ROOT / 'annotation' / 'contracts'
codebook = pd.read_excel(ROOT / 'annotation' / 'regex_annotation_sheet.xlsx', sheet_name='codebook')
ITEMS = ['sell_restriction', 'max_tx_or_wallet_limit', 'blacklist', 'mint_after_deployment']
pd.set_option('display.max_colwidth', 200)
codebook[codebook['item'].isin(ITEMS)][['item', 'definition']]

In [ ]:
def outline(cid):
    """List the functions and modifiers of a contract with their line numbers."""
    for i, line in enumerate((CON / f'{cid}.sol').read_text(errors='ignore').splitlines(), 1):
        if re.match(r'\s*(function|modifier|constructor)\b', line):
            print(f'{i:5d}  {line.strip()[:110]}')

def show(cid, start, end):
    """Print lines start..end of a contract."""
    lines = (CON / f'{cid}.sol').read_text(errors='ignore').splitlines()
    for i in range(start, min(end, len(lines)) + 1):
        print(f'{i:5d}  {lines[i-1]}')

def grep(cid, pattern):
    """Show lines that match a regular expression (case-insensitive)."""
    for i, line in enumerate((CON / f'{cid}.sol').read_text(errors='ignore').splitlines(), 1):
        if re.search(pattern, line, re.I):
            print(f'{i:5d}  {line.strip()[:120]}')

TODAY = ['C53', 'C39', 'C51', 'C22', 'C61']   # five contracts, shown without their labels
outline('C39')

## Part A — Annotate five contracts (45 min, in pairs)

For each contract in `TODAY`, use `outline`, `grep` and `show` to decide Y or N for the four items. Judge what the code **can do**, not what names or comments say; inherited functions count; code that is never called does not. Look in particular at `_transfer`, functions with `onlyOwner`, and mappings of addresses.

Fill in the table below. Do **not** open `annotation_key.csv` or the `annotate` tab yet.

In [ ]:
mine = pd.DataFrame({'id': TODAY, **{it: '' for it in ITEMS}}).set_index('id')
# Example of how to fill in (replace with your own judgement):
# mine.loc['C39', 'blacklist'] = 'Y'
mine

## Part B — Compare with the authors' annotation and with the patterns (20 min)

In [ ]:
ann = pd.read_excel(ROOT / 'annotation' / 'regex_annotation_sheet.xlsx', sheet_name='annotate').set_index('id')
key = pd.read_csv(ROOT / 'annotation' / 'annotation_key.csv').set_index('id')
cmp = ann.loc[TODAY, ITEMS].copy()
cmp['label'] = key.loc[TODAY, 'label'].map({1: 'rug pull', 0: 'legitimate'})
cmp['regex has_max_tx'] = key.loc[TODAY, 'V0_has_max_tx']; cmp['regex has_blacklist'] = key.loc[TODAY, 'V0_has_blacklist']
cmp['regex has_mint'] = key.loc[TODAY, 'V0_has_mint']
cmp

**Question B1.** Where do you disagree with the authors? Re-read the codebook and the `rule_decisions` tab: is the disagreement about the code or about the definition?

**Question B2.** C61 is legitimate, C22 is a reported rug pull. What does that tell you about what source code can and cannot show?

## Part C — Precision and recall of two patterns on all 60 annotated contracts (25 min)

In [ ]:
scored = key[key.scored.astype(bool)].index           # C01 was a training example
def validity(regex_col, item):
    a = ann.loc[scored, item].astype(str).str.upper().eq('Y').values
    r = key.loc[scored, regex_col].astype(int).values == 1
    tp, fp, fn = int((r & a).sum()), int((r & ~a).sum()), int((~r & a).sum())
    return dict(pattern=regex_col, mechanism=item, present=int(a.sum()), TP=tp, FP=fp, FN=fn,
                precision=round(tp / (tp + fp), 2) if tp + fp else np.nan, recall=round(tp / (tp + fn), 2) if tp + fn else np.nan)
pd.DataFrame([validity('V0_has_max_tx', 'max_tx_or_wallet_limit'), validity('V0_has_blacklist', 'blacklist')])

In [ ]:
# Why does the max-tx pattern miss so much? One reason is the word boundary \b:
pat = r'\bmaxtxamount\b|\b_maxtx\b|\bmaxwallet\b'          # the pattern used in the paper (on lower-cased code)
for name in ['maxtxamount', '_maxtxamount', 'maxwalletsize', '_maxwalletsize']:
    print(f'{name:16s} matched: {bool(re.search(pat, name))}')

In Python, `_` counts as a letter, so `\b` does not separate `_maxtxamount` into `_maxtx` + `amount`. And `maxwalletsize` is not `maxwallet` followed by a boundary.

**Exercise C1.** Write a better pattern for transaction or wallet limits and measure its precision and recall on the 60 contracts with `grep`-style matching over the files (template below).

**Exercise C2.** You tuned the pattern on the same 60 contracts you measure it on. What is the problem with reporting that number in a paper, and how would you avoid it?

In [ ]:
def regex_flag(cid, pattern):
    return bool(re.search(pattern, (CON / f'{cid}.sol').read_text(errors='ignore').lower()))
my_pattern = r'maxtx|maxwallet'          # <- edit
a = ann.loc[scored, 'max_tx_or_wallet_limit'].astype(str).str.upper().eq('Y').values
r = np.array([regex_flag(c, my_pattern) for c in scored])
tp, fp, fn = (r & a).sum(), (r & ~a).sum(), (~r & a).sum()
print('precision', round(tp / max(tp + fp, 1), 2), '| recall', round(tp / max(tp + fn, 1), 2))

## Wrap-up (10 min)
The paper found recall of 0.03 for blacklists and 0.04 for transfer locks. If such an indicator gets a small SHAP value, what can you conclude about blacklists as a fraud mechanism?